In [1]:
import pandas as pd
import numpy as np
import joblib
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit
import matplotlib.pyplot as plt

In [2]:
import pandas as pd

def inspect_dataset(path, name, nrows=5):
    print(f"\n=== {name} ===")
    try:
        df = pd.read_csv(path, nrows=nrows)
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        print("\nHead:\n", df.head())
        print("\nInfo:")
        print(df.info())
        print("\nMissing values:\n", df.isna().sum())
    except Exception as e:
        print(f"Error loading {name}: {e}")

# List of files and labels
datasets = {
    "Indian_price.csv": "Indian Wheat Price",
    "global-price.csv": "Global Wheat Price",
    "fao_clean.csv": "FAO Cereals Index",
    "DCOILWTICO.csv": "Oil Price (WTI)",
    "DEXINUS.csv": "USD/INR Exchange Rate",
    "number_of_political_violence_events_by_country-month-year_as-of-20Feb2026.csv": "India Conflict Events",
    "daily_rainfall_at_state_level.csv": "India Rainfall",
    "Fertiliser_price_index.csv": "Fertiliser Price Index",
    "india_import_export.csv": "India Wheat PSD",
    "Global_Policy_Uncertainty_Data.csv": "Global Economic Policy Uncertainty",
    "M2SL.csv": "US M2 Money Supply",
    "india_temperature_dataset.csv": "India Temperature"
}

# Inspect each dataset
for filename, label in datasets.items():
    inspect_dataset(f"finaldatasets/{filename}", label)


=== Indian Wheat Price ===
Shape: (5, 5)
Columns: ['Month', 'Modal Price (?)', 'Min Price (?)', 'Max Price (?)', 'Change (?)']

Head:
     Month  Modal Price (?)  Min Price (?)  Max Price (?)  Change (?)
0  Oct-25          2540.83        2465.48        2599.39      -36.71
1  Sep-25          2577.54        2496.01        2641.01      -10.38
2  Aug-25          2587.93        2509.14        2645.56       59.53
3  Jul-25          2528.40        2442.86        2600.61       21.91
4  Jun-25          2506.49        2419.55        2575.83       -4.68

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Month            5 non-null      object 
 1   Modal Price (?)  5 non-null      float64
 2   Min Price (?)    5 non-null      float64
 3   Max Price (?)    5 non-null      float64
 4   Change (?)       5 non-null      float64
dtypes: float64(4), o

In [3]:
india = pd.read_csv("finaldatasets/Indian_price.csv")
global_wheat = pd.read_csv("finaldatasets/global-price.csv")
fao = pd.read_csv("finaldatasets/fao_clean.csv")
oil = pd.read_csv("finaldatasets/DCOILWTICO.csv")
fx = pd.read_csv("finaldatasets/DEXINUS.csv")
conflict = pd.read_csv("finaldatasets/number_of_political_violence_events_by_country-month-year_as-of-20Feb2026.csv")
rain = pd.read_csv("finaldatasets/daily_rainfall_at_state_level.csv")
fert = pd.read_csv("finaldatasets/Fertiliser_price_index.csv")
gepu = pd.read_csv("finaldatasets/Global_Policy_Uncertainty_Data.csv")
temp = pd.read_csv("finaldatasets/india_temperature_dataset.csv")
psd = pd.read_csv("finaldatasets/india_import_export.csv")

In [4]:
# Clean numeric columns
# Clean ALL numeric PSD columns
numeric_psd_cols = ["Area Harvested", "Production", "Imports", "Exports", "Domestic Consumption"]

for col in numeric_psd_cols:
    psd[col] = (
        psd[col]
        .astype(str)
        .str.replace(",", "")
        .str.replace(" ", "")
        .replace("", "0")
        .astype(float)
    )


# Convert "2000/2001" → 2000
psd["year"] = psd["Year"].str.split("/").str[0].astype(int)

# Expand annual data to monthly
rows = []

for _, row in psd.iterrows():
    year = row["year"]
    months = pd.date_range(start=f"{year}-01-01", end=f"{year}-12-01", freq="MS")
    
    for m in months:
        rows.append({
            "month": m,
            "wheat_production": row["Production"],
            "wheat_imports": row["Imports"],
            "wheat_exports": row["Exports"],
            "wheat_consumption": row["Domestic Consumption"]
        })

psd_monthly = pd.DataFrame(rows)

In [5]:
india["month"] = pd.to_datetime(india["Month"], format="%b-%y")
india = india.rename(columns={"Modal Price (?)": "india_price"})
india = india[["month", "india_price"]]

In [6]:
gw_raw = pd.read_csv("finaldatasets/global-price.csv")
gw_raw.columns
gw_raw.head()

,Month,Price,Change
0,Feb-06,"7,972.50",-
1,Mar-06,"7,758.31",-2.69%
2,Apr-06,"8,106.63",4.49%
3,May-06,"8,771.67",8.20%
4,Jun-06,"8,988.18",2.47%


In [7]:
gw = pd.read_csv("finaldatasets/global-price.csv")

# Clean column names
gw.columns = gw.columns.str.strip()

# Convert Month to datetime
gw["Month"] = pd.to_datetime(gw["Month"], format="%b-%y")

# Clean numeric price (remove commas)
gw["Price"] = (
    gw["Price"]
    .astype(str)
    .str.replace(",", "")
    .astype(float)
)

# Rename for consistency
gw = gw.rename(columns={
    "Month": "month",
    "Price": "global_wheat_price"
})

# Keep only needed columns
gw = gw[["month", "global_wheat_price"]]

gw.head()

,month,global_wheat_price
0,2006-02-01,7972.50
1,2006-03-01,7758.31
2,2006-04-01,8106.63
3,2006-05-01,8771.67
4,2006-06-01,8988.18


In [8]:
fao["month"] = pd.to_datetime(fao["date"])
fao = fao.rename(columns={"Cereals": "fao_cereals"})
fao = fao[["month", "fao_cereals"]]

In [9]:
oil["observation_date"] = pd.to_datetime(oil["observation_date"], format="%d/%m/%Y")

oil_m = (
    oil.resample("ME", on="observation_date").mean()
        .reset_index()
)

# Convert month-end → month-start
oil_m["month"] = oil_m["observation_date"].dt.to_period("M").dt.to_timestamp()

oil_m = oil_m.rename(columns={"DCOILWTICO": "oil_price"})
oil_m = oil_m[["month", "oil_price"]]

In [10]:
oil_m.head(1000)

,month,oil_price
0,2000-01-01,27.259474
1,2000-02-01,29.366000
2,2000-03-01,29.841739
3,2000-04-01,25.722105
4,2000-05-01,28.788182
...,...,...
309,2025-10-01,60.894545
310,2025-11-01,60.062222
311,2025-12-01,57.972273
312,2026-01-01,60.037000


In [11]:
oil_m.isna().sum().sort_values(ascending=False)

month        0
oil_price    0
dtype: int64

In [12]:
fx["observation_date"] = pd.to_datetime(fx["observation_date"])

fx_m = (
    fx.resample("ME", on="observation_date").mean()
        .reset_index()
)

fx_m["month"] = fx_m["observation_date"].dt.to_period("M").dt.to_timestamp()

fx_m = fx_m.rename(columns={"DEXINUS": "usd_inr"})
fx_m = fx_m[["month", "usd_inr"]]

In [13]:
conflict_india = conflict[conflict["COUNTRY"] == "India"].copy()

conflict_india["month"] = pd.to_datetime(
    conflict_india["YEAR"].astype(str) + "-" + conflict_india["MONTH"], 
    format="%Y-%B"
)

conflict_india = conflict_india.rename(columns={"EVENTS": "conflict_events"})
conflict_india = conflict_india[["month", "conflict_events"]]

In [14]:
rain["date"] = pd.to_datetime(rain["date"])

rain_m = (
    rain.groupby(rain["date"].dt.to_period("M"))["actual"]
    .mean()
    .reset_index()
)

rain_m["month"] = rain_m["date"].dt.to_timestamp()
rain_m = rain_m.rename(columns={"actual": "rainfall"})
rain_m = rain_m[["month", "rainfall"]]

In [15]:
fert["observation_date"] = pd.to_datetime(fert["observation_date"], format="%d/%m/%Y")
fert = fert.rename(columns={"observation_date": "month", "PCU3253132531": "fertiliser_index"})
fert = fert[["month", "fertiliser_index"]]

In [16]:
gepu["month"] = pd.to_datetime(
    gepu["Year"].astype(str) + "-" + gepu["Month"].astype(str) + "-01"
)

gepu = gepu.rename(columns={"GEPU_current": "gepu"})
gepu = gepu[["month", "gepu"]]

In [17]:
temp["date"] = pd.to_datetime(temp["date"], format="%d/%m/%Y")

temp_m = (
    temp.groupby(temp["date"].dt.to_period("M"))["average_temperature"]
    .mean()
    .reset_index()
)

temp_m["month"] = temp_m["date"].dt.to_timestamp()
temp_m = temp_m.rename(columns={"average_temperature": "temperature"})
temp_m = temp_m[["month", "temperature"]]

In [18]:
df = india.merge(gw, on="month", how="left") \
         .merge(fao, on="month", how="left") \
         .merge(oil_m, on="month", how="left") \
         .merge(fx_m, on="month", how="left") \
         .merge(conflict_india, on="month", how="left") \
         .merge(rain_m, on="month", how="left") \
         .merge(fert, on="month", how="left") \
         .merge(gepu, on="month", how="left") \
         .merge(temp_m, on="month", how="left") \
         .merge(psd_monthly, on="month", how="left")


df = df.sort_values("month")

In [19]:
df.to_parquet("master_monthly_11_03.parquet")
df.head(100)

,month,india_price,global_wheat_price,fao_cereals,oil_price,usd_inr,conflict_events,rainfall,fertiliser_index,gepu,temperature,wheat_production,wheat_imports,wheat_exports,wheat_consumption
306,2001-01-01,771.00,NaN,53.9,29.585714,46.613810,NaN,NaN,NaN,99.216299,18.730571,69681.0,32.0,3087.0,65126.0
305,2001-03-01,659.63,NaN,52.2,27.244545,46.653182,NaN,NaN,NaN,112.151962,24.397714,69681.0,32.0,3087.0,65126.0
304,2001-04-01,899.84,NaN,51.0,27.490000,46.788571,NaN,NaN,NaN,107.023471,27.390286,69681.0,32.0,3087.0,65126.0
303,2001-05-01,854.90,NaN,51.7,28.629091,46.947273,NaN,NaN,NaN,82.950790,29.495143,69681.0,32.0,3087.0,65126.0
302,2001-06-01,855.46,NaN,51.0,27.599048,47.042381,NaN,NaN,NaN,74.085198,27.842857,69681.0,32.0,3087.0,65126.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
211,2009-01-01,1132.84,11678.13,100.4,41.710000,48.699500,NaN,0.189954,234.1,146.111248,19.262500,80679.0,218.0,58.0,78149.0
210,2009-02-01,1137.05,11059.74,97.2,39.087368,49.248421,NaN,0.212012,184.9,147.703144,21.420000,80679.0,218.0,58.0,78149.0
209,2009-03-01,1122.09,11842.34,97.2,47.939091,51.129091,NaN,0.894190,181.1,132.581749,24.194722,80679.0,218.0,58.0,78149.0
208,2009-04-01,1116.29,11690.12,98.1,49.646667,49.965455,NaN,1.576000,184.2,102.623135,27.471111,80679.0,218.0,58.0,78149.0


In [20]:
df.isna().sum().sort_values(ascending=False)

conflict_events       179
rainfall              127
global_wheat_price     68
temperature            68
fertiliser_index       34
india_price             0
month                   0
usd_inr                 0
oil_price               0
fao_cereals             0
gepu                    0
wheat_production        0
wheat_imports           0
wheat_exports           0
wheat_consumption       0
dtype: int64

In [21]:
(df.isna().sum() / len(df) * 100).sort_values(ascending=False)

conflict_events       58.306189
rainfall              41.368078
global_wheat_price    22.149837
temperature           22.149837
fertiliser_index      11.074919
india_price            0.000000
month                  0.000000
usd_inr                0.000000
oil_price              0.000000
fao_cereals            0.000000
gepu                   0.000000
wheat_production       0.000000
wheat_imports          0.000000
wheat_exports          0.000000
wheat_consumption      0.000000
dtype: float64